In [1]:
# Install dependencies
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Load env variables
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
# Create an API client
from anthropic import Anthropic
client = Anthropic()
model = "claude-sonnet-5"

In [4]:
# basic 'create' function to send message to API
# message = client.messages.create(
#     model=model,
#     max_tokens=1000,
#     messages=[
#         # List of messages send
#         {
#             'role':'user',
#             'content':'What is a car? Answer in one sentence'
#         }
#     ]
# )

# add conversation in to history
# add user message
def add_user_message(messages, text):
    user_message = {'role':'user', 'content':text}
    messages.append(user_message)
# add AI's reply
def add_assistant_message(messages, text):
    assistant_message = {'role':'assistant', 'content':text}
    messages.append(assistant_message)

# Make a request
def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages
    )
    return message.content[0].text

In [5]:
messages = []
add_user_message(messages, "Define quantum computing in one sentence")
answer = chat(messages)
add_assistant_message(messages, answer)
add_user_message(messages, "Write another sentence")
answer = chat(messages)
print(answer)

Instead of using classical bits that are strictly 0 or 1, quantum computers use qubits, which can exist in a combination of states simultaneously, enabling them to explore many possible solutions at once.


In [6]:
# implement simple chat box
# messages = []

# while True:
#     user_input = input('> ')
#     print('>', user_input)

#     add_user_message(messages, user_input)
#     answer = chat(messages)
#     add_assistant_message(messages, answer)

#     print('------')
#     print(answer)
#     print('------')

In [7]:
# Implement Math Tutor Specialist
# responses should: 1. Intially only give hit 2. Walk through solution 3. Show solution for similar problem

# use system_prompt to set tone and style of response
system_prompt="""
you are a patient math tutor. 
Do not directly answer a student's questions.
Guide then to a solution step by step.
"""

def chat(messages, system=None):
    params = {
        "model":model,
        "max_tokens":1000,
        "messages":messages
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params) # dictionary unpacking
    return message.content[0].text

messages = []

add_user_message(messages, "How do I solve 5x + 3 = 2 for x?")
answer = chat(messages, system=system_prompt)
print(answer)

I'd be happy to guide you through this!

First, let's look at the equation: **5x + 3 = 2**

Our goal is to get x by itself on one side of the equation. Right now, x has two things attached to it: it's being multiplied by 5, and there's a 3 being added.

**Question for you:** What operation would you do first to start isolating x? Think about what's "attached" to the 5x term.


In [8]:
# Temperature make LLM answer more creativly, range: 0.0 - 1.0 less creative to more

def chat(messages, system=None, temperature=1.0):
    params = {
        "model":model,
        "max_tokens":1000,
        "messages":messages,
        "temperature":temperature
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params) # dictionary unpacking
    return message.content[0].text

messages = []

add_user_message(messages, "A one sentence movie idea")
answer = chat(messages, temperature=1.0)
print(answer)

AttributeError: 'ThinkingBlock' object has no attribute 'text'

In [ ]:
# Response streaming: send some text so user can see some messages first with out waiting too long
messages = []

add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_011CdJHbhpicdLXyREEXwmC4', container=None, content=[], model='claude-sonnet-5', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=21, output_tokens=1, output_tokens_details=None, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='A', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' cloud-based NoSQL database called "Nimbustore', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='" that automatically shards user

RawMessageDeltaEvent contain actual generated text

In [ ]:
messages = []

add_user_message(messages, "Write a 1 sentence description of a fake database")

with client.messages.stream( # use stream instead of create, use with to make sure connect closed
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end='') # print it like stream
print()
print(stream.get_final_message().content[0].text) # print the entire result out

A cloud-based inventory management database called "StockSync" that tracks product quantities, supplier information, and order histories in real-time across multiple retail store locations.
A cloud-based inventory management database called "StockSync" that tracks product quantities, supplier information, and order histories in real-time across multiple retail store locations.


In [ ]:
messages = []
add_user_message(messages, "Generate a very short event bridge rule as json")
chat(messages)

'Here\'s a very short EventBridge rule in JSON:\n\n```json\n{\n  "source": ["aws.ec2"],\n  "detail-type": ["EC2 Instance State-change Notification"],\n  "detail": {\n    "state": ["running"]\n  }\n}\n```\n\nThis rule triggers when an EC2 instance changes to the "running" state. Let me know if you\'d like a different example (e.g., for S3, custom events, or a scheduled rule)!'

In [ ]:
# We don't want extra symbol like ```json
# new model doent support assistant prefill anymore

# Option 1: use system_prompt directly
messages = []
def chat(messages, system=None, stop_sequences=None):
    params = {
        "model":model,
        "max_tokens":1000,
        "messages":messages
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    message = client.messages.create(**params)
    return message.content[0].text

system_prompt = """
Respond with raw JSON only.
Do not include markdown code fences (no ```json or ```).
Do not include any explanation or extra text before or after the JSON.
"""

add_user_message(messages, "Generate a very short event bridge rule as json.")
# add_assistant_message(messages, '```json') # new model doent support assistant prefill anymore
# answer = chat(messages, stop_sequences=['```'])
answer = chat(messages, system=system_prompt)
print(answer)

{
  "Rules": [
    {
      "Name": "SampleRule",
      "EventPattern": {
        "source": ["aws.ec2"],
        "detail-type": ["EC2 Instance State-change Notification"]
      },
      "State": "ENABLED",
      "Targets": [
        {
          "Id": "1",
          "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction"
        }
      ]
    }
  ]
}


In [ ]:
import json
json.loads(answer.strip())

{'Rules': [{'Name': 'SampleRule',
   'EventPattern': {'source': ['aws.ec2'],
    'detail-type': ['EC2 Instance State-change Notification']},
   'State': 'ENABLED',
   'Targets': [{'Id': '1',
     'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:MyFunction'}]}]}

In [ ]:
message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {"role": "user", "content": "Generate a very short EventBridge rule."}
    ],
    output_config={
        "format": {
            "type": "json_schema",
            "schema": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "source": {"type": "array", "items": {"type": "string"}},
                    "detail-type": {"type": "array", "items": {"type": "string"}},
                    "detail": {
                        "type": "object",
                        "additionalProperties": False
                    }
                },
                "required": ["source", "detail-type"]
            }
        }
    }
)
print(message.content[0].text)

{"source":["custom.myapp"],"detail-type":["Order Placed"]}


In [18]:
rule = json.loads(message.content[0].text)

print(json.dumps(rule, indent=2))

{
  "source": [
    "custom.myapp"
  ],
  "detail-type": [
    "Order Placed"
  ]
}
